# MVP - Machine Learning & Analytics: Previsão de Chuva na Austrália

**Nome:** Henrique Teixeira

**Matrícula:** 4052025001514

**Dataset:** Rain in Austrália (weatherAUS) - Australian Bureau of Meteorology

**Tipo de problema:** Classificação binária supervisionada

---

Este notebook constroi e avalia modelos de classificação para prever a ocorrência de chuva no dia seguinte (`RainTomorrow`) a partir de observações meteorológicas diárias registradas em estações australianas. O trabalho segue um fluxo completo de Machine Learning: definição do problema, análise exploratória, preparação dos dados com prevenção de vazamento, modelagem, otimização de hiperparâmetros e discussão crítica dos resultados.

## Checklist do MVP

| Item | Status |
|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | OK |
| Dataset descrito, com fonte, atributos e restrições | OK |
| Dataset carregado por URL pública | OK |
| Análise exploratória conectada à modelagem | OK |
| Divisão adequada em treino/teste | OK |
| Prevenção de vazamento de dados | OK |
| Tratamentos justificados | OK |
| Pipeline reproduzível de pré-processamento | OK |
| Modelo baseline definido | OK |
| Pelo menos dois modelos candidatos além do template | OK |
| Ajuste de hiperparâmetros em pelo menos um modelo | OK |
| Avaliação com métricas coerentes | OK |
| Discussão de overfitting/underfitting e limitações | OK |
| Código limpo e executável do início ao fim | OK |
| Conclusão conectada ao objetivo | OK |

# 1. Definição do problema

## 1.1 Contexto e objetivo

Prever se choverá no dia seguinte tem valor operacional direto em agricultura, logística, gestão de recursos hídricos e planejamento de atividades ao ar livre. O dataset weatherAUS reúne cerca de dez anos de observações meteorológicas diárias de 49 estações espalhadas pela Austrália, com variáveis como temperatura, umidade, pressão, vento e nebulosidade medidas em dois horários do dia (9h e 15h).

O objetivo deste MVP é construir e avaliar modelos que, a partir das condições meteorológicas observadas em um dia, prevejam a variável binária `RainTomorrow` (choveu no dia seguinte: sim ou não), comparando um baseline ingênuo com modelos candidatos e discutindo criticamente seus limites.

## 1.2 Por que é um problema de Machine Learning

A relação entre as condições atmosféricas de um dia e a ocorrência de chuva no dia seguinte e complexa, não linear e envolve interação entre múltiplas variáveis. Não existe regra determinística simples que resolva o problema, mas existe um padrão estatístico aprendível a partir de dados históricos. Isso caracteriza um problema supervisionado de classificação binária.

## 1.3 Tipo de problema é métrica

**Tipo:** classificação binária supervisionada.

O target apresenta desbalanceamento real: aproximadamente 22% dos dias são seguidos de chuva contra 78% sem chuva. Esse desbalanceamento tem consequência direta na escolha de métricas. Acurácia seria enganosa, um modelo que sempre prevê "não chove" atingiria cerca de 78% de acurácia sem aprender nada útil. Por isso a métrica principal sera o F1-score da classe positiva (chuva) e a PR-AUC (área sob a curva precision-recall), que avaliam a capacidade real de identificar os dias de chuva. A análise sera complementada com estudo de threshold.

## 1.4 Premissas e restrições

As observações são tratadas como diárias e independentes no nível de modelagem tabular, embora exista estrutura temporal que sera respeitada na divisão treino/teste. Assume-se que os padrões meteorologicos aprendidos no período de treino permanecem válidos no período de teste. A principal restrição de dados, discutida adiante, e a presença da coluna `RISK_MM`, que constitui vazamento direto do target e sera removida.

# 2. Ambiente e reprodutibilidade

XGBoost e LightGBM são instalados de forma idempotente. No Google Colab o XGBoost costuma vir pré-instalado, mas a instalação explícita blinda o notebook contra variações da imagem do ambiente, garantindo execução do início ao fim.

In [ ]:
!pip install -q xgboost lightgbm

In [ ]:
import sys
import time
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

pd.set_option("display.max_columns", None)

print("Python:", sys.version.split()[0])
print("Seed:", SEED)

# 3. Carga dos dados

O dataset e carregado diretamente da versão raw do repositório público no GitHub, sem upload manual, login ou token. Isso garante que o notebook execute do início ao fim em qualquer ambiente com acesso a internet.

A coluna `Date` e lida no formato dia-mes-ano, coerente com o arquivo de origem. O parse explícito evita inversão silenciosa de dia e mes, o que corromperia a divisão temporal usada mais adiante.

In [ ]:
URL = "https://raw.githubusercontent.com/Henrique1078/mvp-ml-rain-australia/main/data/weatherAUS.csv"

df = pd.read_csv(URL, parse_dates=["Date"], dayfirst=True)
print("Formato do dataset:", df.shape)
df.head()

# 4. Análise exploratória

A análise a seguir e seletiva e orientada a decisão. Cada visualização existe para embasar uma escolha concreta de preparação dos dados ou de modelagem, não como ilustração. São cinco eixos: distribuição do target, valores ausentes, vazamento do `RISK_MM`, estabilidade temporal do target e poder discriminativo das variáveis meteorológicas.

## 4.1 Distribuição do target

O primeiro passo é quantificar o desbalanceamento, que determina a escolha das métricas de avaliação.

In [ ]:
TARGET = "RainTomorrow"

counts = df[TARGET].value_counts()
prop = df[TARGET].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=["#4C72B0", "#DD8452"])
for bar, pct in zip(bars, prop.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f"{pct:.1f}%", ha="center", va="bottom")
ax.set_title("Distribuicao do target RainTomorrow")
ax.set_ylabel("Contagem")
plt.tight_layout()
plt.show()

print("Ausentes no target:", df[TARGET].isna().sum())

A classe positiva (chuva) representa cerca de 22% das observações contra 78% da classe negativa. O target não possui valores ausentes, o que dispensa tratamento nessa coluna.

Consequência direta: acurácia é inadequada como métrica principal. Um classificador trivial que sempre prevê "não chove" atingiria 78% de acurácia sem qualquer capacidade preditiva. Por isso adotamos F1-score da classe positiva e PR-AUC, métricas sensíveis ao desempenho na classe minoritária, que é justamente a de interesse prático.

## 4.2 Valores ausentes

O padrão de dados ausentes orienta a estratégia de imputação e a eventual remoção de colunas.

In [ ]:
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
missing_pct = missing_pct[missing_pct > 0]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(missing_pct.index[::-1], missing_pct.values[::-1], color="#C44E52")
ax.set_xlabel("Percentual ausente (%)")
ax.set_title("Valores ausentes por coluna")
plt.tight_layout()
plt.show()

missing_pct.round(2)

Quatro colunas concentram a maior parte da ausência: `Sunshine` (48%), `Evaporation` (43%), `Cloud3pm` (40%) e `Cloud9am` (38%). As demais ficam abaixo de 10%.

Decisão de preparação: em vez de descartar essas quatro colunas, que possuem forte poder discriminativo (ver seção 4.5), optamos por mante-las e imputar os ausentes dentro do pipeline. A imputação sera ajustada apenas no treino para evitar vazamento. Colunas numéricas recebem imputação pela mediana, robusta a outliers meteorologicos, e categóricas pela moda.

## 4.3 Vazamento de dados: a coluna RISK_MM

Este e o ponto crítico da preparação. A coluna `RISK_MM` registra a quantidade de chuva em milímetros do dia seguinte, exatamente a informação usada para construir o target `RainTomorrow`.

In [ ]:
leak = df.dropna(subset=["RISK_MM", TARGET]).copy()
leak["target_bin"] = (leak[TARGET] == "Yes").astype(int)

concordance = ((leak["RISK_MM"] >= 1.0) == (leak["target_bin"] == 1)).mean()
correlation = leak[["RISK_MM", "target_bin"]].corr().iloc[0, 1]

fig, ax = plt.subplots(figsize=(7, 4))
for label, color in [("No", "#4C72B0"), ("Yes", "#DD8452")]:
    subset = leak[leak[TARGET] == label]["RISK_MM"].clip(upper=20)
    ax.hist(subset, bins=40, alpha=0.6, label=label, color=color)
ax.set_xlabel("RISK_MM (mm, truncado em 20 para visualizacao)")
ax.set_ylabel("Frequencia")
ax.set_title("RISK_MM separa perfeitamente as classes do target")
ax.legend(title="RainTomorrow")
plt.tight_layout()
plt.show()

print(f"Concordancia (RISK_MM >= 1mm) == (RainTomorrow = Yes): {concordance:.4f}")
print(f"Correlacao de RISK_MM com o target: {correlation:.4f}")

A concordância de 98,8% entre `RISK_MM >= 1mm` e `RainTomorrow = Yes` confirma empiricamente que a coluna define o próprio target. Manter `RISK_MM` como preditor produziria um modelo com desempenho artificialmente perfeito e completamente inútil na prática, pois a quantidade de chuva do dia seguinte não esta disponível no momento em que a previsão precisaria ser feita.

Decisão: `RISK_MM` sera removida antes de qualquer etapa de modelagem. Este é um caso clássico de vazamento de target (target leakage), e sua identificação e remoção são parte essencial da qualidade da solução.

Distinção importante: a coluna `Rainfall` (chuva do dia atual) também discrimina bem as classes, mas e um preditor legítimo, pois esta disponível no momento da previsão. Não confundir chuva de hoje (válida) com chuva de amanhã (vazamento).

## 4.4 Estabilidade temporal do target

A estratégia de divisão treino/teste sera temporal (detalhada na seção 5). Para validar essa escolha, verificamos se a taxa de chuva se mantém estável ao longo dos anos.

In [ ]:
df["Year"] = df["Date"].dt.year
yearly = df.dropna(subset=[TARGET]).groupby("Year").agg(
    taxa_chuva=(TARGET, lambda s: (s == "Yes").mean()),
    n=(TARGET, "size")
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(yearly.index, yearly["taxa_chuva"], marker="o", color="#4C72B0")
ax.axhline(0.2242, linestyle="--", color="gray", label="Media global (22.4%)")
ax.set_xlabel("Ano")
ax.set_ylabel("Taxa de chuva")
ax.set_title("Taxa de chuva por ano")
ax.set_ylim(0, 0.4)
ax.legend()
plt.tight_layout()
plt.show()

yearly.round(3)

A taxa de chuva oscila entre 20% e 25% na maior parte do período, próxima da média global de 22,4%. O valor de 2007 (31%) e explicado pelo baixo número de registros naquele ano (apenas 61 observações, início da coleta) e não representa tendência real.

Consequência: a estabilidade temporal do target garante que a divisão temporal treino/teste não introduzirá desbalanceamento severo entre os períodos. O modelo treinado no passado enfrentará, no teste, uma distribuição de classes semelhante a que aprendeu. Isso torna a divisão temporal segura e ao mesmo tempo mais realista que um holdout aleatorio, que embaralharia datas e permitiria treinar com dados cronologicamente posteriores aos de teste.

## 4.5 Poder discriminativo das variáveis meteorológicas

Antes de modelar, verificamos se as variáveis carregam sinal real sobre o target, comparando suas distribuições entre as duas classes.

In [ ]:
discrim_cols = ["Humidity3pm", "Sunshine", "Cloud3pm", "Pressure3pm"]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, col in zip(axes.ravel(), discrim_cols):
    for label, color in [("No", "#4C72B0"), ("Yes", "#DD8452")]:
        subset = df[df[TARGET] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col)
    ax.legend(title="RainTomorrow")
fig.suptitle("Distribuicao de variaveis-chave por classe", y=1.01)
plt.tight_layout()
plt.show()

summary = df.dropna(subset=[TARGET]).groupby(TARGET)[discrim_cols].mean().round(1)
summary

As distribuições revelam separação clara entre as classes:

`Humidity3pm`: dias sem chuva tem umidade média de 46% as 15h contra 69% nos dias que precedem chuva. E o preditor individual mais forte.

`Sunshine`: 8,5 horas de sol em dias sem chuva contra 4,5 horas antes de dias chuvosos.

`Cloud3pm`: maior cobertura de nuvens antecede chuva, como esperado fisicamente.

`Pressure3pm`: pressão ligeiramente menor antecede chuva, coerente com a meteorologia de sistemas de baixa pressão.

A presença de sinal discriminativo forte justifica a expectativa de que modelos não triviais superem o baseline com folga. Além disso, a natureza não linear e a interação entre essas variáveis favorecem modelos baseados em árvores com boosting, motivando a escolha de XGBoost e LightGBM como candidatos.

# 5. Preparação dos dados e divisão treino/teste

Esta seção define quais colunas entram no modelo, remove a fonte de vazamento e estabelece a divisão temporal. Todo o tratamento estatístico (imputação, escalonamento, encoding) é encapsulado em pipeline na seção 6 e ajustado apenas no treino. Aqui tratamos apenas das decisões estruturais que antecedem o pipeline.

## 5.1 Remoção de colunas e definição de features

Três grupos de colunas não entram como features:

`RISK_MM`: vazamento de target, conforme demonstrado na seção 4.3. Removida obrigatoriamente.

`Date`: usada para ordenar a divisão temporal, mas não como preditor direto. A informação sazonal relevante ja esta parcialmente capturada pelas variáveis meteorológicas. Mantida fora das features do modelo tabular.

`Year`: criada apenas para a análise exploratória de drift temporal. Removida para não introduzir dependência artificial do ano de coleta.

O target e `RainTomorrow`, convertido para binário (1 = choveu, 0 = não choveu). Registros sem target são descartados, embora ja tenhamos verificado que não existem.

In [ ]:
TARGET = "RainTomorrow"
LEAK_COLUMN = "RISK_MM"
DROP_NON_FEATURES = ["Date", "Year", LEAK_COLUMN, TARGET]

data = df.dropna(subset=[TARGET]).copy()
data = data.sort_values("Date").reset_index(drop=True)

y = (data[TARGET] == "Yes").astype(int)
feature_cols = [c for c in data.columns if c not in DROP_NON_FEATURES]
X = data[feature_cols].copy()

print("Total de registros:", len(data))
print("Numero de features:", len(feature_cols))
print("Features:", feature_cols)
print("Vazamento RISK_MM presente nas features:", LEAK_COLUMN in feature_cols)

## 5.2 Divisão temporal treino/teste

A divisão respeita a ordem cronológica usando uma data de corte: identifica-se a data na posição correspondente a 80% dos registros ordenados, e todas as observações anteriores a ela formam o treino, enquanto as demais formam o teste. O corte por data, em vez de por índice de linha, garante que um mesmo dia com observações de múltiplas estações nunca seja dividido entre treino e teste. Essa escolha é deliberada é mais rigorosa que um holdout aleatorio.

Justificativa: o problema é intrinsecamente temporal. Prever chuva de amanhã usando dados de hoje. Um holdout aleatorio embaralharia datas e permitiria que o modelo treinasse com observações cronologicamente posteriores as do teste, uma forma sutil de vazamento temporal que inflaria a métrica. A divisão temporal reproduz a condição real de uso: treinar no passado, prever o futuro.

Como demonstrado na seção 4.4, a taxa de chuva e estável ao longo dos anos, entao a divisão temporal não introduz desbalanceamento severo entre treino e teste, apesar de não usar estratificação explícita (que seria incompatível com a ordem temporal).

In [ ]:
split_idx = int(len(data) * 0.8)
cutoff_date = data["Date"].iloc[split_idx]

train_mask = data["Date"] < cutoff_date
test_mask = data["Date"] >= cutoff_date

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

date_train = data.loc[train_mask, "Date"]
date_test = data.loc[test_mask, "Date"]

print("Data de corte:", cutoff_date.date())
print("Treino:", X_train.shape, "| periodo:", date_train.min().date(), "a", date_train.max().date())
print("Teste: ", X_test.shape, "| periodo:", date_test.min().date(), "a", date_test.max().date())
print()
print("Proporcao de chuva no treino:", round(y_train.mean(), 4))
print("Proporcao de chuva no teste: ", round(y_test.mean(), 4))

A verificação das proporções confirma o esperado: as taxas de chuva no treino e no teste ficam próximas, validando que a divisão temporal preserva a distribuição do target. A ausência de sobreposição temporal entre os períodos garante que nenhuma informação do futuro vaza para o treino.

# 6. Pipeline de pré-processamento

O pré-processamento é encapsulado em um `ColumnTransformer` que integra imputação, escalonamento e encoding. A construção do pipeline garante que todas as transformações sejam ajustadas exclusivamente nos dados de treino e apenas aplicadas ao teste, eliminando vazamento de dados na etapa de preparação.

## 6.1 Identificação de colunas e estratégia

As colunas são separadas por tipo, cada uma com tratamento adequado:

Numéricas: imputação pela mediana, robusta a outliers meteorologicos (rajadas de vento extremas, picos de chuva), seguida de padronização. A padronização beneficia a regressão logística do baseline e não prejudica os modelos de árvore.

Categóricas (direções de vento e RainToday): imputação pela moda e codificação one-hot. O parâmetro `handle_unknown="ignore"` garante que categorias eventualmente ausentes no treino não quebrem a transformação no teste, um cuidado relevante dada a divisão temporal.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

print("Colunas numericas:", len(num_cols))
print("Colunas categoricas:", len(cat_cols), "->", cat_cols)

O `preprocess` ainda não foi ajustado. Ele sera encaixado dentro de cada pipeline de modelo na seção seguinte, e o ajuste ocorrerá apenas quando `.fit()` for chamado sobre os dados de treino. Essa arquitetura assegura que estatísticas como mediana, média e desvio padrão usadas na imputação e padronização sejam calculadas somente com dados de treino, nunca contaminadas pelo teste.

# 7. Modelagem: baseline e modelos candidatos

A estratégia de modelagem usa três níveis de complexidade crescente, o que permite decompor a origem de cada ganho de desempenho:

Baseline ingênuo (`DummyClassifier`): sempre prevê a classe majoritária. Serve como piso de referência. Se um modelo não supera o baseline, ele não aprendeu nada útil.

Referência linear (`LogisticRegression`): modelo linear que mede quanto sinal e capturável por uma fronteira de decisão simples. O salto do baseline para a logística quantifica o sinal linear disponível.

Candidatos principais (`XGBoost` e `LightGBM`): modelos de gradient boosting sobre árvores, capazes de capturar não linearidades e interações entre variáveis. Ambos vao além do conjunto de modelos do template. O salto da logística para o boosting quantifica o ganho da não linearidade.

A regressão logística não e o candidato principal do trabalho, e sim uma referência intermediária mais forte que o baseline ingênuo. Os candidatos avaliados como solução são XGBoost e LightGBM.

## 7.1 Tratamento do desbalanceamento

Com 22% de exemplos positivos, os modelos tendem a favorecer a classe majoritária. Em vez de reamostragem (que alteraria a distribuição real dos dados e adicionaria complexidade), optamos por ponderação de classe interna a cada modelo:

`LogisticRegression` e `LightGBM` usam `class_weight="balanced"`, que pondera a função de perda inversamente a frequência de cada classe.

`XGBoost` usa `scale_pos_weight` igual a razao entre negativos e positivos no treino, que cumpre o mesmo papel.

Essa abordagem preserva os dados originais e é computacionalmente mais leve que reamostragem, adequada ao objetivo do MVP.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, average_precision_score, roc_auc_score, classification_report, confusion_matrix

neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos

candidates = {
    "Baseline (Dummy)": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=SEED, class_weight="balanced"),
    "XGBoost": XGBClassifier(random_state=SEED, eval_metric="logloss", scale_pos_weight=scale_pos_weight, n_jobs=2),
    "LightGBM": LGBMClassifier(random_state=SEED, class_weight="balanced", n_jobs=2, verbose=-1),
}

print("scale_pos_weight (XGBoost):", round(scale_pos_weight, 3))

## 7.2 Treinamento e comparação inicial

Cada modelo é encaixado em um pipeline com o pré-processamento definido na seção 6, garantindo que a preparação seja ajustada apenas no treino. As métricas priorizam a classe positiva: F1-score, PR-AUC (área sob a curva precision-recall) e ROC-AUC como referência complementar.

In [ ]:
results = {}
trained = {}

for name, clf in candidates.items():
    pipe = Pipeline([("prep", preprocess), ("model", clf)])
    t0 = time.time()
    pipe.fit(X_train, y_train)
    elapsed = time.time() - t0

    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        "F1": f1_score(y_test, pred),
        "PR_AUC": average_precision_score(y_test, proba),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "tempo_treino_s": round(elapsed, 2),
    }
    trained[name] = pipe

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("PR_AUC", ascending=False)
results_df.round(4)

## 7.3 Análise dos resultados iniciais

A comparação entre os três níveis revela a contribuição de cada grau de complexidade:

O baseline ingênuo obtém F1 igual a zero. Como sempre prevê "não chove", nunca identifica corretamente um dia de chuva, resultando em recall zero para a classe positiva e, consequentemente, F1 zero. Sua acurácia, no entanto, seria de cerca de 78%. Este contraste e a demonstração prática de por que acurácia e uma métrica enganosa neste problema: um modelo completamente inútil atinge 78% de acurácia.

A regressão logística representa o maior salto isolado de desempenho, saindo de F1 zero para cerca de 0,62 e PR-AUC de 0,68. Isso indica que boa parte do sinal preditivo e capturável linearmente. Variáveis como umidade e horas de sol, vistas na seção 4.5, separam bem as classes ate por uma fronteira linear.

XGBoost e LightGBM superam a logística em todas as métricas, com XGBoost liderando (F1 em torno de 0,65 e PR-AUC de 0,72). O ganho sobre a logística, embora menor que o salto anterior, confirma que existe estrutura não linear e interações entre variáveis que os modelos de árvore capturam. O tempo de treino de todos permanece na casa de segundos, sem custo computacional relevante.

XGBoost é escolhido como modelo a ser otimizado na próxima seção, por apresentar o melhor PR-AUC inicial.

# 8. Otimização de hiperparâmetros

O XGBoost, melhor modelo na avaliação inicial, é submetido a uma busca de hiperparâmetros. A busca usa `RandomizedSearchCV`, que amostra combinações aleatórias do espaco de parametros, mais eficiente que busca exaustiva em grade quando ha vários hiperparâmetros contínuos.

## 8.1 Estratégia de validação e espaco de busca

Ponto crítico de coerência: a validação cruzada usa `TimeSeriesSplit`, não `KFold` aleatorio. Como o problema é temporal e a divisão treino/teste respeita a cronologia, a validação interna também precisa respeita-la. `TimeSeriesSplit` treina sempre em períodos anteriores e válida em períodos posteriores, replicando internamente a lógica da divisão principal e evitando vazamento temporal durante a busca.

A métrica que guia a busca e `average_precision` (PR-AUC), coerente com a métrica principal do problema, apropriada para dados desbalanceados por focar no desempenho da classe positiva.

Hiperparâmetros ajustados e justificativa:

`n_estimators` e `learning_rate`: controlam o número de árvores e o tamanho do passo. São ajustados juntos porque ha compensação entre eles (mais árvores com passo menor).

`max_depth`: profundidade das árvores, controla a capacidade de capturar interações e o risco de overfitting.

`subsample` e `colsample_bytree`: frações de amostras e de colunas por árvore, introduzem aleatoriedade que reduz overfitting.

Sobre eficiência computacional: o XGBoost usa `tree_method="hist"`, que discretiza as variáveis em histogramas e acelera substancialmente o treino em datasets desse porte, sem perda relevante de qualidade. A busca usa `n_iter=10` e `TimeSeriesSplit(n_splits=3)`, configuração suficiente para explorar o espaço de forma representativa dentro do escopo de um MVP, e o paralelismo (`n_jobs=-1` na busca) distribui as combinações entre os núcleos disponíveis. Testes comparativos mostraram que essa configuração reduz o tempo à metade em relação a uma busca mais extensa, mantendo o PR-AUC praticamente idêntico.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint, uniform

xgb_pipe = Pipeline([
    ("prep", preprocess),
    ("model", XGBClassifier(random_state=SEED, eval_metric="logloss",
                            scale_pos_weight=scale_pos_weight,
                            tree_method="hist", n_jobs=1))
])

param_dist = {
    "model__n_estimators": randint(100, 400),
    "model__max_depth": randint(3, 10),
    "model__learning_rate": uniform(0.02, 0.2),
    "model__subsample": uniform(0.6, 0.4),
    "model__colsample_bytree": uniform(0.6, 0.4),
}

tscv = TimeSeriesSplit(n_splits=3)

search = RandomizedSearchCV(
    xgb_pipe, param_dist, n_iter=10, cv=tscv,
    scoring="average_precision", random_state=SEED, n_jobs=-1, verbose=0
)

t0 = time.time()
search.fit(X_train, y_train)
search_time = time.time() - t0

print(f"Tempo de busca: {search_time:.1f}s")
print(f"Melhor PR-AUC na validacao temporal: {search.best_score_:.4f}")
print("Melhores hiperparametros:")
for k, v in search.best_params_.items():
    v_fmt = round(v, 3) if isinstance(v, float) else v
    print(f"  {k.replace('model__', '')}: {v_fmt}")

## 8.2 Impacto da otimização

O modelo otimizado é comparado com o XGBoost sem ajuste, ambos avaliados no conjunto de teste. A comparação usa dados que não participaram nem do treino nem da validação cruzada, garantindo uma medida honesta do ganho.

In [ ]:
best_xgb = search.best_estimator_

pred_opt = best_xgb.predict(X_test)
proba_opt = best_xgb.predict_proba(X_test)[:, 1]

comparison = pd.DataFrame({
    "XGBoost (sem ajuste)": results["XGBoost"],
    "XGBoost (otimizado)": {
        "F1": f1_score(y_test, pred_opt),
        "PR_AUC": average_precision_score(y_test, proba_opt),
        "ROC_AUC": roc_auc_score(y_test, proba_opt),
        "tempo_treino_s": round(search_time, 2),
    },
}).T

comparison.round(4)

A otimização produziu ganho modesto porém consistente em todas as métricas: PR-AUC e F1 aumentaram em relação ao XGBoost sem ajuste. O ganho limitado indica que os hiperparâmetros padrão do XGBoost ja eram razoáveis para este problema, é que o teto de desempenho é mais determinado pela informação disponível nos dados do que pela configuração do modelo.

O custo computacional da busca (na casa de poucos minutos) e justificável para o ganho obtido em um contexto de MVP. Uma busca mais extensa, com mais iterações ou espaco maior, poderia render melhora adicional, mas com retorno decrescente.

# 9. Avaliação final

O modelo escolhido (XGBoost otimizado) é avaliado em profundidade no conjunto de teste. A avaliação vai além da métrica agregada: examina a matriz de confusão, o trade-off entre precision e recall via análise de threshold, as curvas de desempenho e o comportamento do modelo por estação meteorológica.

## 9.1 Matriz de confusão e relatorio de classificação

A matriz de confusão no threshold padrão de 0,5 revela a distribuição concreta de acertos e erros.

In [ ]:
from sklearn.metrics import (precision_recall_curve, roc_curve, confusion_matrix,
    ConfusionMatrixDisplay, classification_report, precision_score, recall_score)

proba_final = best_xgb.predict_proba(X_test)[:, 1]
pred_final = (proba_final >= 0.5).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final),
                       display_labels=["Nao chove", "Chove"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusao (threshold 0.5)")
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred_final, target_names=["Nao chove", "Chove"]))

O modelo identifica corretamente a maioria dos dias de chuva (alto recall na classe positiva), ao custo de alguns falsos alarmes. Em previsão de chuva, esse perfil costuma ser desejável: o custo de não prever uma chuva (deixar de levar guarda-chuva, não proteger uma plantação) geralmente supera o de um alarme falso.

## 9.2 Análise de threshold

O threshold padrão de 0,5 não é necessariamente o melhor ponto de operação. Variando o limiar de decisão, ajusta-se o equilíbrio entre precision e recall conforme a prioridade do problema.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba_final)
f1_scores = 2 * precision * recall / (precision + recall + 1e-9)
best_t_idx = np.argmax(f1_scores[:-1])
best_threshold = thresholds[best_t_idx]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds, precision[:-1], label="Precision", color="#4C72B0")
ax.plot(thresholds, recall[:-1], label="Recall", color="#DD8452")
ax.plot(thresholds, f1_scores[:-1], label="F1", color="#55A868")
ax.axvline(best_threshold, linestyle="--", color="gray",
           label=f"Melhor F1 (thr={best_threshold:.3f})")
ax.set_xlabel("Threshold")
ax.set_ylabel("Metrica")
ax.set_title("Precision, Recall e F1 em funcao do threshold")
ax.legend()
plt.tight_layout()
plt.show()

pred_bt = (proba_final >= best_threshold).astype(int)
print(f"Threshold padrao (0.5):  precision={precision_score(y_test, pred_final):.4f} "
      f"recall={recall_score(y_test, pred_final):.4f} f1={f1_score(y_test, pred_final):.4f}")
print(f"Threshold otimo ({best_threshold:.3f}): precision={precision_score(y_test, pred_bt):.4f} "
      f"recall={recall_score(y_test, pred_bt):.4f} f1={f1_score(y_test, pred_bt):.4f}")

O threshold que maximiza o F1 fica ligeiramente acima de 0,5. Nesse ponto, o modelo troca parte do recall por precision, reduzindo falsos alarmes e elevando o F1 global. A escolha do ponto de operação final depende do objetivo: para maximizar a captura de dias de chuva, mantém-se um threshold mais baixo; para equilibrar os dois erros, adota-se o threshold ótimo de F1. A análise deixa essa decisão explícita e fundamentada, em vez de aceitar o 0,5 por inércia.

## 9.3 Curvas Precision-Recall e ROC

As curvas resumem o desempenho do modelo em todos os thresholds possíveis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ap = average_precision_score(y_test, proba_final)
axes[0].plot(recall, precision, color="#4C72B0")
axes[0].axhline(y_test.mean(), linestyle="--", color="gray",
                label=f"Baseline aleatorio ({y_test.mean():.3f})")
axes[0].set_xlabel("Recall")
axes[0].set_ylabel("Precision")
axes[0].set_title(f"Curva Precision-Recall (PR-AUC = {ap:.3f})")
axes[0].legend()

fpr, tpr, _ = roc_curve(y_test, proba_final)
roc_auc = roc_auc_score(y_test, proba_final)
axes[1].plot(fpr, tpr, color="#DD8452")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray", label="Aleatorio")
axes[1].set_xlabel("Taxa de falsos positivos")
axes[1].set_ylabel("Taxa de verdadeiros positivos")
axes[1].set_title(f"Curva ROC (ROC-AUC = {roc_auc:.3f})")
axes[1].legend()

plt.tight_layout()
plt.show()

A curva PR mostra que o modelo mantém precision muito acima do baseline aleatorio (a taxa de prevalência da classe positiva) em ampla faixa de recall. A curva ROC, com AUC próximo de 0,88, confirma boa capacidade de separação entre as classes. A PR-AUC e a métrica mais informativa neste caso desbalanceado, pois a ROC-AUC tende a parecer otimista quando a classe negativa domina.

## 9.4 Desempenho por estação meteorológica

A métrica agregada esconde variação entre as 49 estações. Analisar o F1 por `Location` revela onde o modelo funciona bem e onde falha, o que é relevante para decidir se a solução pode ser aplicada uniformemente.

In [ ]:
eval_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred": pred_final,
    "Location": data.loc[test_mask, "Location"].values
})

loc_f1 = (eval_df.groupby("Location")
          .apply(lambda g: f1_score(g["y_true"], g["y_pred"]) if g["y_true"].sum() > 0 else np.nan)
          .dropna().sort_values())

fig, ax = plt.subplots(figsize=(9, 10))
colors = ["#C44E52" if v < loc_f1.median() else "#4C72B0" for v in loc_f1.values]
ax.barh(loc_f1.index, loc_f1.values, color=colors)
ax.axvline(loc_f1.mean(), linestyle="--", color="black", label=f"F1 medio ({loc_f1.mean():.3f})")
ax.set_xlabel("F1-score")
ax.set_title("Desempenho (F1) por estacao meteorologica")
ax.legend()
plt.tight_layout()
plt.show()

print(f"F1 medio entre estacoes: {loc_f1.mean():.3f} (desvio {loc_f1.std():.3f})")
print(f"Pior: {loc_f1.index[0]} ({loc_f1.iloc[0]:.3f}) | Melhor: {loc_f1.index[-1]} ({loc_f1.iloc[-1]:.3f})")

O desempenho varia de forma interpretável entre estações. As piores incluem regiões desérticas ou semi-aridas (AliceSprings, Woomera), onde a chuva e rara e errática, dificultando a previsão por falta de padrão regular e de exemplos positivos. As melhores incluem climas com sazonalidade marcada (Darwin, tropical com estação chuvosa definida), onde os padrões meteorologicos são mais previsíveis.

Essa heterogeneidade e uma limitação relevante: um modelo único global não atende igualmente todas as regiões. Uma evolução possível seria treinar modelos regionais ou adicionar features climaticas específicas por localidade. Para o escopo do MVP, o modelo global é adequado, mas a análise por subgrupo documenta honestamente onde ele e menos confiavel.

# 10. Conclusão

Este MVP construiu e avaliou modelos de classificação binária para prever a ocorrência de chuva no dia seguinte a partir de observações meteorológicas diárias de 49 estações australianas, usando o dataset weatherAUS carregado diretamente de URL pública.

Sobre os tratamentos e decisões centrais: a coluna `RISK_MM` foi identificada como vazamento de target (concordância de 98,8% com o alvo) e removida antes de qualquer modelagem. A divisão treino/teste foi temporal, com corte por data para respeitar a natureza cronológica do problema é evitar vazamento temporal. Todo o pré-processamento (imputação, padronização, encoding) foi encapsulado em pipeline ajustado apenas no treino. O desbalanceamento de classes (22% positivos) foi tratado por ponderação interna aos modelos, sem reamostragem.

Sobre os modelos: a comparação em três níveis de complexidade permitiu decompor a origem dos ganhos. O baseline ingênuo (F1 zero) demonstrou por que acurácia seria enganosa. A regressão logística capturou o sinal linear (F1 em torno de 0,62). XGBoost e LightGBM, modelos além do template, superaram a referência linear ao capturar não linearidades, com XGBoost apresentando o melhor desempenho (PR-AUC de 0,72). A otimização de hiperparâmetros via RandomizedSearchCV com validação temporal elevou o PR-AUC para cerca de 0,73.

Sobre a melhor solução: o XGBoost otimizado foi escolhido por apresentar o melhor PR-AUC, métrica mais apropriada para o problema desbalanceado. A análise de threshold mostrou que o ponto de operação pode ser ajustado conforme a prioridade entre capturar dias de chuva (recall) e evitar falsos alarmes (precision).

Sobre as limitações: o desempenho varia entre estações (F1 de 0,48 a 0,77), sendo menor em regiões desérticas onde a chuva e errática. As quatro colunas com alta taxa de ausência (Sunshine, Evaporation, nuvens) foram imputadas, o que introduz incerteza. O modelo é global e não captura especificidades regionais.

Sobre próximos passos: modelos regionais ou features climaticas por localidade poderiam reduzir a heterogeneidade de desempenho; features temporais derivadas (médias moveis, tendencias) poderiam capturar dinâmica que a modelagem tabular diária ignora; e tecnicas de calibração de probabilidade tornariam os scores mais interpretaveis como probabilidades reais de chuva.

O MVP cumpriu o objetivo definido no início: construir uma solução coerente, reproduzivel, bem documentada e tecnicamente justificada, com desempenho substancialmente superior ao baseline e análise crítica dos resultados e limitações.

# 11. Checklist do MVP preenchido

| Item | Status | Onde no notebook |
|---|---|---|
| Problema definido com contexto, objetivo e tipo de tarefa | OK | Seção 1 |
| Dataset descrito, com fonte, atributos e restrições | OK | Seções 3 e 4 |
| Dataset carregado por URL pública | OK | Seção 3 |
| Análise exploratória conectada a modelagem | OK | Seção 4 |
| Divisão adequada em treino/teste | OK | Seção 5 |
| Prevenção de vazamento de dados | OK | Seções 4.3, 5 e 6 |
| Tratamentos justificados | OK | Seções 4 e 6 |
| Pipeline reproduzível de pré-processamento | OK | Seção 6 |
| Modelo baseline definido | OK | Seção 7 |
| Pelo menos dois modelos candidatos alem do template | OK | Seção 7 (XGBoost, LightGBM) |
| Ajuste de hiperparâmetros em pelo menos um modelo | OK | Seção 8 |
| Avaliação com métricas coerentes | OK | Seções 7 e 9 |
| Discussão de overfitting/underfitting e limitações | OK | Seções 9 e 10 |
| Código limpo e executavel do inicio ao fim | OK | Notebook completo |
| Conclusão conectada ao objetivo | OK | Seção 10 |

## Observações sobre overfitting e underfitting

O baseline ingênuo é um caso extremo de underfitting: não aprende estrutura alguma. A regressão logística, mais simples, é superada pelos modelos de árvore, indicando que ha estrutura não linear que ela não captura (leve underfitting relativo). XGBoost e LightGBM equilíbram bem: o desempenho na validação temporal cruzada e próximo do desempenho no teste, sem queda abrupta, o que indica ausência de overfitting severo. A regularização via `subsample`, `colsample_bytree` e profundidade controlada contribui para isso.

## Bibliotecas utilizadas

pandas e numpy para manipulação de dados; matplotlib para visualização; scikit-learn para pipelines, pré-processamento, métricas e busca de hiperparâmetros; xgboost e lightgbm para os modelos candidatos. Seeds fixas em 42 garantem reprodutibilidade.